## Read the data

In [35]:
import numpy as np
import pandas as pd
import sys
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)
METHOD = "hmm_pomegranate"
EXPERIMENT= "paper2_core_clustering"

In [36]:
DATA_PATH_BASE = 'G:/My Drive/Work/ITMO/data'
# DATA_PATH_BASE = 'D:/Google Drive/Work/ITMO/data' #windows
# DATA_PATH_BASE = '/home/denklewer/Data' #ubuntu

DATA_PATH = f'experiment_results/paper2_core_clustering/peptide_cores' # work laptop
APPROACH = "DECOY_PATHS"

In [37]:
data_df = pd.read_csv(f"{DATA_PATH}/extractd_cores.csv", index_col=0, sep=';')


In [38]:
DATA_PATH

'experiment_results/paper2_core_clustering/peptide_cores'

In [39]:
ALLELES = list(data_df['allele'].unique())
selected_alleles = ALLELES
models_to_read = ['v']

# Read models

In [40]:
## PARAMETERS
RUNS = 20
# RUNS = 1
TOTAL_SPLITS = 1
SELECTED_RUN = 0
SELECTED_RUN = set(np.arange(0, RUNS))
# SELECTED_RUN = {0}
#
SELECTED_SPLIT = 0
MODELS_DESCRIPTIONS = {
           "class_I_simple_clustering": {
        "path": "experiment_results/paper2_core_clustering/reconstructed_models/",
        "states": 15,
        "params_string": "60-min_iters-4-aaas",
        "combination_size": 2,
        "alleles": [
                # 'HLA-DRB1*01:01'
                    'HLA-A*01:01',
                    'HLA-A*03:01'
                    ],
        "target_lengths": [8, 9,10, 11, 12,13, 14],
        "core_start_state": [15,16],
        "core_end_state": [17,18],
        "need_to_delete": [],
        "skip_states": [],

    },
    
    
               "class_I_simple_clustering_simulated": {
        "path": "experiment_results/paper2_core_clustering/reconstructed_models/",
        "states": 15,
        "params_string": "60-min_iters-4-aaas",
        "combination_size":1,
        "alleles": [
             "dummy_allele"
                # 'HLA-DRB1*01:01'

                    ],
        "target_lengths": [ 9,10, 11],
        "core_start_state": [15,16],
        "core_end_state": [17,18],
        "need_to_delete": [],
        "skip_states": [],

    },




}

In [41]:
MODELS_DESCRIPTIONS.keys()

dict_keys(['class_I_simple_clustering', 'class_I_simple_clustering_simulated'])

In [42]:
selected_models = [
                 #'single_allele_imbalanced',
                 #'multiple_allele_imbalanced',
                    # 'single_allele_balanced',
                   # 'multiple_allele_balanced',
    # "single_allele_balanced_class2",
    #     "single_allele_balanced_class2_joiners",
    # "single_allele_with_cycle_base_model",
    #  "anchor_model_5_single_allele_balanced_data_attempt3",
    'class_I_simple_clustering_simulated'
    # 'multiple_allele_balanced_longer_training'              
]


In [43]:
from hmm_logic_methods import load_model

In [44]:
import itertools

name_to_run = dict()
name_to_split = dict()
name_to_type = dict()
name_to_model_root = dict()
name_to_target_lengths = dict()
selected_binding_models = list()
per_name_models = dict()


def populate_models_infos(model_name, run_num, split_num, model_path, model_type, target_lengths ):
    selected_binding_models.append(model_name)

    name_to_run[model_name] = run_num
    name_to_split[model_name] = split_num
    name_to_model_root[model_name] = model_path
    assert model_type in ["bw", "v", "bwv", "bw-it", "bwv-it", "v-it"] or "mixed" in model_type
    name_to_type[model_name] = model_type
    name_to_target_lengths[model_name] = target_lengths

for model in selected_models:
    model_description = MODELS_DESCRIPTIONS[model]
    combination_size = model_description['combination_size']
    model_alleles = model_description['alleles']
    MODELS_PATH = model_description['path']
    params_string = model_description['params_string']
    states = model_description['states']
    TARGET_LENGTHS = model_description['target_lengths']
    target_length_string = "-".join([str(l) for l in TARGET_LENGTHS])
    MIXED_ALLELES = list()

    for combination in itertools.combinations(sorted(model_alleles), combination_size):
        allele_name = "+".join(combination)
        # allele_name = f"{allele_name1}+{allele_name2}"
        MIXED_ALLELES.append(allele_name)

    for allele_name in MIXED_ALLELES:
        target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_').replace('/', '_')
        for run_num_b in range(RUNS):
            for split_num_b in range(TOTAL_SPLITS):
                #bw
                if 'bw' in models_to_read:
                    bw_model_name = f"{run_num_b}_run-{target_length_string}-{states}-st-baum-welch-alg-{params_string}_model-{target_allele_name}-{split_num_b}"
                    print(bw_model_name)
                    populate_models_infos(bw_model_name, run_num_b,split_num_b, MODELS_PATH, f"bw-it-mixed({target_allele_name})", TARGET_LENGTHS)
                if 'bwv' in models_to_read:
                    #bwv
                    bwv_model_name = f"{run_num_b}_run-{target_length_string}-{states}-st-baum-welch-alg-{params_string}_model-{target_allele_name}-{split_num_b}_plus_viterbi"
                    populate_models_infos(bwv_model_name, run_num_b,split_num_b, MODELS_PATH, f"bwv-it-mixed({target_allele_name})", TARGET_LENGTHS)
                if 'v' in models_to_read:
                    #v
                    v_model_name = f"{run_num_b}_run-{target_length_string}-{states}-st-viterbi-alg-{params_string}_model-{target_allele_name}-{split_num_b}"
                    populate_models_infos(v_model_name, run_num_b,split_num_b, MODELS_PATH, f"v-it-mixed({target_allele_name})", TARGET_LENGTHS)



for model_name in selected_binding_models:
    per_name_models[model_name] =  load_model(name_to_model_root[model_name], model_name)

In [45]:
list(per_name_models.keys())

['0_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0',
 '1_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0',
 '2_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0',
 '3_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0',
 '4_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0',
 '5_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0',
 '6_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0',
 '7_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0',
 '8_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0',
 '9_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0',
 '10_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0',
 '11_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0',
 '12_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-

## Select run and split

In [46]:
for model_key in list(per_name_models.keys()):
    if name_to_split[model_key] != SELECTED_SPLIT:
        del per_name_models[model_key]
    elif type(SELECTED_RUN) == set:
        if name_to_run[model_key] not in SELECTED_RUN:
            del per_name_models[model_key]
    elif name_to_run[model_key] != SELECTED_RUN:
        del per_name_models[model_key]
per_name_models = dict(reversed(sorted(per_name_models.items(), key=lambda item: item[0])))

In [47]:
list(per_name_models.keys())

['9_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0',
 '8_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0',
 '7_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0',
 '6_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0',
 '5_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0',
 '4_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0',
 '3_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0',
 '2_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0',
 '1_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0',
 '19_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0',
 '18_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0',
 '17_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0',
 '16_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele

In [48]:
import importlib
import hmm_logic_methods
importlib.reload(hmm_logic_methods)

<module 'hmm_logic_methods' from 'C:\\Projects\\MHC_predictor\\experiments\\core_clustering_first_paper\\../..\\hmm_logic_methods.py'>

## Extract paths 
We have two approaches: all possible paths and decoy paths

* precision used to crop transitions
* min and max path lengths are  min/max of (TARGET_LENGTHS)

In [49]:
from hmm_logic_methods import get_all_paths_with_probs, get_model_representation

# code for saving paths do model folder
def save_paths_df(paths_df, model, file_name, save_index=False):
    print(f"model {model.name}")
    
    TARGET_PATH_TO_FILES = f"{name_to_model_root[model.name]}{model.name}/"
    paths_df.to_csv("{}/{}".format(TARGET_PATH_TO_FILES,file_name),sep=";", index=save_index)

In [50]:
# exctract paths
if APPROACH == 'total_paths':
    for model_key, model in per_name_models.items():
        print(model_key)
        TARGET_LENGTHS = name_to_target_lengths[model_key]
        paths_df , edge_to_prob = get_all_paths_with_probs(model, cutoff=max(TARGET_LENGTHS), precision=3, min_path_length=min(TARGET_LENGTHS), max_path_length=max(TARGET_LENGTHS))
        save_paths_df(paths_df, model,"total_paths_df.csv")
        break

In [51]:
data_df.peptide.str.len().value_counts()

9     8306
11    1446
12     426
13     212
8      192
14      74
15      32
Name: peptide, dtype: int64

# Extract only viterbi paths for decoy for models
* we need some data here

## Preparation: get decoy data

In [52]:
TARGET_LENGTHS = list(data_df.core_peptide.str.len().unique())
TARGET_LENGTHS

[9, 13, 11, 12, 8, 14, 15]

In [53]:
from mhcgnomes import parse
# decoy data
def defineOrganism(allele_name):
    parse_result = parse(allele_name)
    if parse_result.is_mouse:
        return "Mouse"
    elif parse_result.is_human:
        return "Human"
    elif selected_models[0].contains("simulated"):
        return "Human"
    else:
        return None

def defineClass(allele_name):
    parse_result = parse(allele_name)
    if parse_result.has_mhc_class:
        if parse_result.is_class1:
            return 'I'
        elif parse_result.is_class2:
            print('Class II data')
            return 'II'
        elif selected_models[0].constains("class_I"):
            print('Simulated class I')
            return "I"
        elif selected_models[0].constains("class_II"):
            print('Simulated class II')
            return "II"
        else:
            print("Not class I or class II???!")
    else:
        print("Result without class")
    return None

per_allele_decoy_peptides = dict()
for allele_name in ALLELES:
    organism =  defineOrganism(allele_name).lower()
    class_to_use = defineClass(allele_name)
    ## Hard coded here
    class_to_use = 'I'
    DECOY_DATA_PATH = f"{DATA_PATH_BASE}\decoy\sampled_decoy_data_{organism}_class_{class_to_use}.csv"
    decoy_df =  pd.read_csv(DECOY_DATA_PATH)
    decoy_df['length'] = decoy_df['peptide'].str.len()
    per_allele_decoy_peptides[allele_name] = decoy_df.copy()[decoy_df['length'].isin(TARGET_LENGTHS)]['peptide'].values

peptides_for_analysis = np.unique(np.concatenate([value for key, value in per_allele_decoy_peptides.items()
                                                  if key in selected_alleles]))

## Calculate viterbi paths

In [54]:
decoy_df['length'].value_counts()

8     200000
9     200000
10    200000
11    200000
12    200000
13    200000
14    200000
Name: length, dtype: int64

In [55]:
sample_model = per_name_models[list(per_name_models.keys())[0]]
peptide = peptides_for_analysis[0]
logp, path = sample_model.viterbi(peptide)
print("Viterbi for Sequence: '{}'  -- Log Probability: {} -- Path: {}".format(''.join( peptide ), logp, " ".join( state.name for idx, state in path[1:-1] ) ))

Viterbi for Sequence: 'AAAAAAAA'  -- Log Probability: -35.01997396723765 -- Path: s015 s000 s005 s008 s009 s010 s012 s018


In [56]:
per_name_models.keys()

dict_keys(['9_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0', '8_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0', '7_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0', '6_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0', '5_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0', '4_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0', '3_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0', '2_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0', '1_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0', '19_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0', '18_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0', '17_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0', '16_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0

In [57]:
%%time
from hmm_logic_methods import get_all_viterbi_paths_for_model_and_sequences_mark_core

# exctract paths
for model_key, model in per_name_models.items():
    print(model_key)
    TARGET_LENGTHS = name_to_target_lengths[model_key]
    paths_df = get_all_viterbi_paths_for_model_and_sequences_mark_core(model, peptides_for_analysis, per_model_important_states=None)
    save_paths_df(paths_df, model,"viterbi_paths_df.csv")


# df_bw_paths = get_all_viterbi_paths_for_model_and_sequences(model_bw, peptides_for_analysis)
# df_bwv_paths = get_all_viterbi_paths_for_model_and_sequences(model_bwv, peptides_for_analysis)
# df_v_paths = get_all_viterbi_paths_for_model_and_sequences(model_v, peptides_for_analysis)

9_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0
Model 9_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0
Get viterbi paths for each peptide...


100%|█████████████████████████████████████████████████████████████████████| 1200000/1200000 [00:35<00:00, 33617.59it/s]


Subselect unique paths (if unique path argument passed)..
	Unique paths: 349116 


100%|██████████████████████████████████████████████████████████████████████| 349116/349116 [00:03<00:00, 103887.27it/s]


model 9_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0
8_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0
Model 8_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0
Get viterbi paths for each peptide...


100%|█████████████████████████████████████████████████████████████████████| 1200000/1200000 [00:39<00:00, 30253.24it/s]


... (71 outputs left out)

100%|█████████████████████████████████████████████████████████████████████| 1200000/1200000 [00:37<00:00, 32105.68it/s]


Subselect unique paths (if unique path argument passed)..
	Unique paths: 189586 


100%|██████████████████████████████████████████████████████████████████████| 189586/189586 [00:01<00:00, 109662.70it/s]


model 0_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0
CPU times: total: 14min 35s
Wall time: 14min 39s


In [58]:
paths_df[paths_df.peptide_cnt > 200]

,path,norm_tr_prob,len,norm_viterbi_prob,norm_bw_prob,peptide_cnt
0,grouped_model-start>s016>s002>s005>s006>s010>s...,-1.195372,8,-4.233949,-4.005638,1064
1,grouped_model-start>s016>s002>s005>s008>s006>s...,-0.767233,9,-3.882527,-3.703015,1033
11,grouped_model-start>s016>s002>s005>s008>s006>s...,-0.869975,11,-3.813271,-3.639916,213
26,grouped_model-start>s016>s002>s005>s008>s006>s...,-0.872139,11,-3.769537,-3.595686,217
29,grouped_model-start>s016>s002>s005>s008>s006>s...,-0.737832,9,-3.830275,-3.658689,883
...,...,...,...,...,...,...
67725,grouped_model-start>s016>s001>s005>s007>s006>s...,-0.789786,9,-3.330027,-3.237057,378
67938,grouped_model-start>s016>s001>s005>s008>s006>s...,-0.708028,9,-3.420858,-3.303644,411
67947,grouped_model-start>s016>s001>s005>s008>s006>s...,-0.719521,9,-3.443168,-3.321214,346
67979,grouped_model-start>s016>s001>s003>s008>s006>s...,-0.693705,9,-3.365246,-3.244650,291


# Same thing but extract paths for training data

In [34]:
for selected_allele in model_alleles:
    total_peptides = np.concatenate([per_allele_per_kfold_per_length_binders_train[selected_allele][0][target_length] for target_length in TARGET_LENGTHS])
    print(len(total_peptides), selected_allele)

NameError: name 'per_allele_per_kfold_per_length_binders_train' is not defined

In [21]:
total_peptides = np.concatenate([per_allele_per_kfold_per_length_binders_train[allele_name][0][target_length] for target_length in TARGET_LENGTHS for allele_name in model_alleles])

In [22]:
from hmm_logic_methods import get_all_viterbi_paths_for_model_and_sequences_mark_core

# exctract paths
for model_key, model in per_name_binding_model.items():
    print(model_key)
    TARGET_LENGTHS = name_to_target_lengths[model_key]
    paths_df = get_all_viterbi_paths_for_model_and_sequences_mark_core(model, total_peptides, per_model_important_states, unique_paths=False)
    #    paths_df = get_all_viterbi_paths_for_model_and_sequences(model, peptides_for_analysis)

    save_paths_df(paths_df, model,"viterbi_training_paths_df.csv", save_index=True)

4_run-13-14-15-16-17-18-25-st-baum-welch-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0
Model 4_run-13-14-15-16-17-18-25-st-baum-welch-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0
Get viterbi paths for each peptide...


100%|██████████| 20925/20925 [00:03<00:00, 6475.34it/s]


Subselect unique paths (if unique path argument passed)..
	Unique paths: 20249 


100%|██████████| 20249/20249 [00:01<00:00, 12497.54it/s]


model 4_run-13-14-15-16-17-18-25-st-baum-welch-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0
3_run-13-14-15-16-17-18-25-st-baum-welch-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0
Model 3_run-13-14-15-16-17-18-25-st-baum-welch-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0
Get viterbi paths for each peptide...


100%|██████████| 20925/20925 [00:03<00:00, 6019.41it/s]


... (11 outputs left out)

100%|██████████| 20925/20925 [00:03<00:00, 6321.20it/s]


Subselect unique paths (if unique path argument passed)..
	Unique paths: 20249 


100%|██████████| 20249/20249 [00:01<00:00, 11987.28it/s]


model 0_run-13-14-15-16-17-18-25-st-baum-welch-alg-30-min_iters_model-HLA_DPA1_01_03_DPB1_04_01+HLA_DRB1_01_01-0


In [23]:
paths_df

,path,simplified_path,core_start,core_end,norm_tr_prob,len,norm_viterbi_prob,norm_bw_prob,peptide_cnt
RSTYNHLSSWLTD,grouped_model-start>s027>s025>s003>s006>s008>s...,"[0.0, 0.0, 0.0, 1.32, 0.0, 0.0, 1.32, 0.0, 1.3...",3,8,-0.791855,13,-3.368854,-3.047072,1
KMYFALIDTKAYK,grouped_model-start>s027>s025>s003>s006>s008>s...,"[0.0, 0.0, 0.0, 1.27, 0.0, 0.0, 1.27, 0.0, 1.2...",3,8,-0.612463,13,-3.249273,-2.982877,1
PGLARQAPKPRKQ,grouped_model-start>s025>s001>s002>s005>s011>s...,"[0.0, 0.0, 1.35, 0.0, 0.0, 1.35, 0.0, 1.35, 0....",2,7,-0.843387,13,-3.493799,-3.140942,1
AEILELAGNAARD,grouped_model-start>s027>s025>s002>s005>s011>s...,"[0.0, 0.0, 1.25, 0.0, 0.0, 1.25, 0.0, 1.25, 0....",2,7,-0.715628,13,-2.989283,-2.737880,1
SYVSLKAPLTKPL,grouped_model-start>s025>s002>s005>s011>s012>s...,"[0.0, 1.29, 0.0, 0.0, 1.29, 0.0, 1.29, 0.0, 0....",1,6,-1.020668,13,-3.196945,-2.905789,1
...,...,...,...,...,...,...,...,...,...
GIHPTIISESFQKALEKG,grouped_model-start>s027>s025>s001>s004>s000>s...,"[0.0, 0.0, 0.0, 0.0, 0.0, 1.19, 0.0, 0.0, 1.19...",5,10,-0.587096,18,-2.890393,-2.699106,1
PREDRATWKSNYFLKIIQ,grouped_model-start>s027>s025>s001>s004>s001>s...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.35999999...",7,12,-0.845403,18,-3.338233,-2.978306,1
ISEKHDVVFLITKYGYIH,grouped_model-start>s027>s025>s001>s004>s001>s...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.359...",8,13,-0.861269,18,-3.437878,-3.073706,1
LPEPPPLASNLFAQLAEN,grouped_model-start>s027>s025>s001>s004>s001>s...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.29, 0.0, 0.0,...",6,11,-0.774668,18,-3.050593,-2.761140,1


# Same thing but generate random paths from model

In [37]:
data_df

,allele,simplified_path,peptide,core_start,core_end,core_len,peptide_len,core_peptide,core_path
GAPRGDKEELTPQK,HLA-DQA1*05:01/DQB1*02:01,[ 0. 6.24 14.4 14.4 15.41 15.41 15.41 15....,GAPRGDKEELTPQK,2,10,9,14,PRGDKEELT,[14.4 14.4 15.41 15.41 15.41 15.41 15.41 15....
AYDGKDYLALNEDLR,HLA-DQA1*05:01/DQB1*02:01,[ 0. 0. 0. 2.01 3.07 12.19 15.2 15....,AYDGKDYLALNEDLR,5,13,9,15,DYLALNEDL,[12.19 15.2 15.2 15.2 15.2 15.2 15.2 13....
VDDDFQTIMSAIEEGKG,HLA-DQA1*05:01/DQB1*02:01,[ 0. 0. 0. 1.01 4.05 7.11 11.33 13....,VDDDFQTIMSAIEEGKG,6,14,9,17,TIMSAIEEG,[11.33 13.39 15.44 15.44 15.44 15.44 14.43 11....
TLTISSLQAEDVAV,HLA-DQA1*05:01/DQB1*02:01,[ 0. 3.14 6.22 15.37 15.37 15.37 15.37 15....,TLTISSLQAEDVAV,3,11,9,14,ISSLQAEDV,[15.37 15.37 15.37 15.37 15.37 15.37 15.37 12....
NPQTQTEERFR,HLA-DQA1*05:01/DQB1*02:01,[ 3.13 15.34 15.34 15.34 15.34 15.34 15.34 15....,NPQTQTEERFR,1,9,9,11,PQTQTEERF,[15.34 15.34 15.34 15.34 15.34 15.34 15.34 15....
...,...,...,...,...,...,...,...,...,...
RTFQKWAAVVVPSGEE,HLA-DQA1*05:01/DQB1*02:01,[ 0. 0. 3.03 8.04 12.05 15.05 15.05 15....,RTFQKWAAVVVPSGEE,3,12,10,16,QKWAAVVVPS,[ 8.04 12.05 15.05 15.05 15.05 15.05 15.05 15....
QPTIPIVGILAG,HLA-DQA1*05:01/DQB1*02:01,[ 0. 4.05 15.25 15.25 15.25 15.25 15.25 15....,QPTIPIVGILAG,2,10,9,12,TIPIVGILA,[15.25 15.25 15.25 15.25 15.25 15.25 15.25 15....
TDVLILPEEVEWIK,HLA-DQA1*05:01/DQB1*02:01,[ 0. 2.1 5.22 8.29 15.45 15.45 15.45 15....,TDVLILPEEVEWIK,3,11,9,14,LILPEEVEW,[ 8.29 15.45 15.45 15.45 15.45 15.45 15.45 13....
SWNLNGLQADLSSF,HLA-DQA1*05:01/DQB1*02:01,[ 1.07 4.09 9.13 14.27 15.33 15.33 15.33 15....,SWNLNGLQADLSSF,2,10,9,14,NLNGLQADL,[ 9.13 14.27 15.33 15.33 15.33 15.33 15.33 14....


In [38]:
from hmm_logic_methods import get_all_viterbi_paths_for_model_and_sequences_mark_core

# exctract paths
for model_key, model in per_name_models.items():
    print(model_key)
    TARGET_LENGTHS = name_to_target_lengths[model_key]
    paths_df = get_all_viterbi_paths_for_model_and_sequences_mark_core(model, None, per_model_important_states=None, unique_paths=True, generate_peptides=True)
    #    paths_df = get_all_viterbi_paths_for_model_and_sequences(model, peptides_for_analysis)

    save_paths_df(paths_df, model,"random_paths_df.csv", save_index=True)


9_run-11-12-13-14-15-16-17-18-19-35-st-viterbi-alg-30-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0
Model 9_run-11-12-13-14-15-16-17-18-19-35-st-viterbi-alg-30-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0
Get viterbi paths for each peptide...


  0%|                                                                                        | 0/50000 [00:00<?, ?it/s]C:\Projects\MHC_predictor\experiments\core_identification\hmm_logic_methods.py:1682: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  peptide_arr, path = model.sample(path=True)
100%|██████████████████████████████████████████████████████████████████████████| 50000/50000 [00:12<00:00, 4100.58it/s]


Subselect unique paths (if unique path argument passed)..
	Unique paths: 32093 


100%|████████████████████████████████████████████████████████████████████████| 32093/32093 [00:00<00:00, 136729.71it/s]


model 9_run-11-12-13-14-15-16-17-18-19-35-st-viterbi-alg-30-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0
8_run-11-12-13-14-15-16-17-18-19-35-st-viterbi-alg-30-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0
Model 8_run-11-12-13-14-15-16-17-18-19-35-st-viterbi-alg-30-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0
Get viterbi paths for each peptide...


100%|██████████████████████████████████████████████████████████████████████████| 50000/50000 [00:12<00:00, 4113.83it/s]


... (51 outputs left out)

100%|██████████████████████████████████████████████████████████████████████████| 50000/50000 [00:14<00:00, 3481.65it/s]


Subselect unique paths (if unique path argument passed)..
	Unique paths: 31125 


100%|████████████████████████████████████████████████████████████████████████| 31125/31125 [00:00<00:00, 145743.84it/s]


model 0_run-11-12-13-14-15-16-17-18-19-35-st-viterbi-alg-30-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0


In [25]:
l =5 